# Descarga de los Financial Statement Data Sets de la SEC

Descarga un zip por trimestre en `zips/`, sin descomprimir.

El período llega hasta `2024q2` porque el zip se nombra por la fecha de presentación. El 10-K de 2023 se presenta en 2024.

La SEC rechaza la descarga (error 403) si el `User-Agent` no tiene un email de contacto.

In [1]:
import os
import sys
import zipfile
from pathlib import Path

import requests


if "google.colab" in sys.modules:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")

    CANDIDATOS = [
        Path("/content/drive/MyDrive/Colab Notebooks/Diplodatos MAI Grupo 6"),
        Path("/content/drive/MyDrive/Diplodatos MAI Grupo 6"),
    ]
    WORKING_DIR = next((p for p in CANDIDATOS if p.exists()), None)
    if WORKING_DIR is None:
        raise FileNotFoundError(
            "No encuentro la carpeta del grupo. ¿Agregaste el acceso directo "
            "en 'Mi unidad' desde 'Compartido conmigo'?"
        )
else:
    WORKING_DIR = Path.cwd()

In [2]:
START, END = 2018, 2024
quarters = [
    f"{y}q{q}"
    for y in range(START, END + 1)  # loop through years
    for q in range(1, 5)  # loop through quarters
]
quarters[:4]  # first 4 quarters


['2018q1', '2018q2', '2018q3', '2018q4']

### ACLARACION: 

- El URL es el del link de descarga de los archivos que aparece en la pagina https://www.sec.gov/data-research/sec-markets-data/financial-statement-data-sets.
- El URL en el navegador funciona solo, pero para descargarlo a travez de un script hay que identificarse de alguna forma. 

In [3]:
URL = "https://www.sec.gov/files/dera/data/financial-statement-data-sets/{}.zip"
EMAIL = "ejemplo@ejemplo.com"  # no se verifica el correo, pero es necesario para descargar! (con ejemplo@ejemplo.com funciona)
HEADERS = {"User-Agent": f"Diplodatos UNC {EMAIL}"}

OUT_DIR = f"{WORKING_DIR}/zips"
os.makedirs(OUT_DIR, exist_ok=True)


for quarter in quarters:
    file = f"{OUT_DIR}/{quarter}.zip"

    # Si un zip ya existe, no se descarga otra vez.
    if os.path.exists(file):
        print(f"El archivo {quarter}.zip: ya existe, no se descarga otra vez.")
        continue

    url = URL.format(quarter)
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()  # error if the SEC does not answer 200

    with open(file, "wb") as f:
        f.write(response.content)

    print(f"{quarter}.zip descargado  ({os.path.getsize(file) / 1e6:.0f} MB)")

2018q1.zip descargado  (93 MB)
2018q2.zip descargado  (92 MB)
2018q3.zip descargado  (87 MB)
2018q4.zip descargado  (63 MB)
2019q1.zip descargado  (96 MB)
2019q2.zip descargado  (90 MB)
2019q3.zip descargado  (90 MB)
2019q4.zip descargado  (99 MB)
2020q1.zip descargado  (97 MB)
2020q2.zip descargado  (82 MB)
2020q3.zip descargado  (88 MB)
2020q4.zip descargado  (90 MB)
2021q1.zip descargado  (98 MB)
2021q2.zip descargado  (91 MB)
2021q3.zip descargado  (97 MB)
2021q4.zip descargado  (103 MB)
2022q1.zip descargado  (104 MB)
2022q2.zip descargado  (97 MB)
2022q3.zip descargado  (98 MB)
2022q4.zip descargado  (111 MB)
2023q1.zip descargado  (114 MB)
2023q2.zip descargado  (117 MB)
2023q3.zip descargado  (118 MB)
2023q4.zip descargado  (120 MB)
2024q1.zip descargado  (124 MB)
2024q2.zip descargado  (119 MB)
2024q3.zip descargado  (118 MB)
2024q4.zip descargado  (123 MB)


Verificación: cada zip tiene los cuatro archivos.

In [4]:
# leemos la info de todos los zip, vemos que en cada uno esten los 4 archivos que esperamos, y si no, tiramos un error.

for quarter in quarters:
    with zipfile.ZipFile(f"{OUT_DIR}/{quarter}.zip") as z:
        missing = {"sub.txt", "num.txt", "pre.txt", "tag.txt"} - set(z.namelist())

    assert not missing, f"{quarter}: missing {missing}"

# vemos cuanto ocupan todos los zips juntos, y cuantos zips tenemos.


print("Cantidad de archivos descargados:", len(quarters))
total = sum(os.path.getsize(f"{OUT_DIR}/{q}.zip") for q in quarters)
print(f"Tamaño total: {total / 1e6:.2f} MB")


Cantidad de archivos descargados: 28
Tamaño total: 2817.58 MB
